In [1]:
import numpy as np
from skimage.io import imread
from skimage import img_as_float
from sklearn.cluster import KMeans

In [2]:
# Функция для вычисления PSNR
def psnr(original, reconstructed):
    mse = np.mean((original - reconstructed) ** 2)
    if mse == 0:
        return float('inf')
    max_val = 1.0  # т.к. изображение в [0, 1]
    return 20 * np.log10(max_val) - 10 * np.log10(mse)

In [3]:
# 1. Загружаем и нормализуем
image = img_as_float(imread('parrots.jpg'))
height, width, channels = image.shape

# 2. Преобразование в матрицу объекты-признаки
pixels = image.reshape(-1, 3)

best_n_clusters = None

# 5. Перебор числа кластеров от 1 до 20
for n_clusters in range(1, 21):
    # 3. Обучение и заливка
    kmeans = KMeans(n_clusters=n_clusters, init='k-means++', random_state=241, n_init=10)
    labels = kmeans.fit_predict(pixels)
    
    # Создаем массивы для восстановленных изображений
    reconstructed_mean = np.zeros_like(pixels)
    reconstructed_median = np.zeros_like(pixels)
    
    for cluster in range(n_clusters):
        mask = (labels == cluster)
        cluster_pixels = pixels[mask]
        
        # Среднее
        mean_color = np.mean(cluster_pixels, axis=0)
        reconstructed_mean[mask] = mean_color
        
        # Медиана
        median_color = np.median(cluster_pixels, axis=0)
        reconstructed_median[mask] = median_color
    
    # Восстановление формы изображения
    image_mean = reconstructed_mean.reshape(height, width, channels)
    image_median = reconstructed_median.reshape(height, width, channels)
    
    # 4. Вычисление PSNR для обоих вариантов
    psnr_mean = psnr(image, image_mean)
    psnr_median = psnr(image, image_median)
    
    best_psnr = max(psnr_mean, psnr_median)
    
    if best_psnr > 20:
        best_n_clusters = n_clusters
        break


print(f"Минимальное количество кластеров: {best_n_clusters}")

Минимальное количество кластеров: 11


In [4]:
with open('answer.txt', 'w') as f:
    f.write(str(best_n_clusters))